# A3 · corrección telúrica — notebook de análisis (`debug`)

**Objeto:** ROXs12b  |  **Run:** `ROXs12b_realigned`  |  **Spec:** [`docs/spec_A3_codex_telluric.md`](../../../docs/spec_A3_codex_telluric.md)

Este notebook **no llama a la cadena**: rehace la decisión de A3 aquí dentro, con el código a la vista, para que puedas **probar, cambiar y ajustar sin tocar `musepipe`**. El notebook de auditoría equivalente es [`../A3_telluric.ipynb`](../A3_telluric.ipynb), que sí lee el QC de la etapa.

Cómo está montado, y por qué:

1. **Perillas** arriba del todo, con el valor que usa la cadena para este run.
2. **Las funciones numéricas, copiadas literalmente** de `musepipe`. Se copian (en vez de importarse) para que puedas editarlas: todo lo que viene después usa estos nombres locales.
3. **Chequeo de deriva** — avisa si `musepipe` cambió y esta copia se quedó atrás.
4. El proceso **paso a paso**, cada uno con su diagnóstico.
5. **Comparación con el producto de la cadena**: con las perillas por defecto debe salir *idéntico*; en cuanto cambias algo, te dice qué se movió y dónde.

### La pregunta que este notebook viene a cerrar

La misma banda, O₂ B, medida sobre el espectro de la primaria, da **~7 %** en las dos reducciones de una noche y **0.59 %** en el cubo multi-noche de la cadena canónica — que por eso decidió `not_needed_shallow` y **no** corrigió. Mientras eso no se explique, la corrección telúrica de la cadena no es citable como «medida y descartada»: lo único medido es que en *ese* cubo la banda es superficial.

Se contrastan tres hipótesis (§7, §8, §9), cada una con su número:

1. el DRS ya quitó las bandas, exposición a exposición, y 0.59 % es **residuo**;
2. combinar 29 exposiciones a masas de aire distintas **difumina** la banda;
3. medir con la **mediana sobre toda la banda** diluye un núcleo estrecho.

> Lo que NO se copia: la resolución del cubo de entrada, la comprobación del entorno de `molecfit` y el escritor del QC — son E/S de la etapa, no su matemática. Y lo que este notebook **no** repite del de auditoría: el porqué de `STD_TELLURIC` frente a `molecfit`, el volcado campo a campo de los dos esquemas de QC y la figura de la curva de transmisión.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

# Resolución de las figuras EN PANTALLA. `savefig` guarda a 300 dpi, pero
# lo que se ve dentro del notebook lo fija el backend inline, que va a 100
# dpi por defecto y sale borroso. `retina` dobla los píxeles sin cambiar el
# tamaño aparente; fuera de IPython no hace nada y queda el rcParam.
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
# El objeto se DERIVA del run (cadena declarada en su config), no se
# escribe: un literal aquí haría que un objeto nuevo heredase el nombre
# del primero, que es lo que vigila tests/test_no_hardcoded_target.py.
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)

def resolver(p):
    """Ruta declarada en un QC -> ruta usable.

    Los QC de A3 guardan unas rutas absolutas y otras relativas a la RAÍZ
    del repo, y el cwd de un notebook es su propia carpeta. Anclar aquí
    evita el fallo que rompió el A3 de auditoría (traspaso 07-29 §6.1).
    """
    if not p:
        return None
    q = Path(p)
    return q if q.is_absolute() else (ROOT / q)

print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)
print('stages :', SD)


## 1 · Perillas

Salen del **config resuelto de la etapa**, no del `config.json` crudo. A3 es la etapa donde esto más importa: es anterior a la convención `musepipe.stages`, así que la mitad de sus perillas vive en los *defaults de argparse* y la otra mitad como literales dentro de las funciones numéricas (el umbral del 3 %, las bandas laterales del continuo, los bordes de las bandas). `stage00t_config_from_run` las reúne en un sitio; copiarlas a mano es exactamente cómo se consigue un notebook que no reproduce la cadena.

Dos avisos que la celda imprime y conviene leer:

- **ningún fichero de config las contiene**: hoy todas salen del default de la etapa, así que un cambio de default cambia lo que reproduce este notebook;
- `a3_science_needs_red_continuum` vale `True` aquí y `False` en el `store_true` de `main`. Los tres QC en disco registran `true`, y `decide_telluric` cortocircuita a `not_needed_science` si es falso: la bandera se pasó. El resolutor reproduce lo que corrió, no lo que el default sugiere.


In [ ]:
from musepipe.reduction.telluric import stage00t_config_from_run

# `project_root=ROOT` no es opcional: musepipe resuelve rutas contra el cwd, y
# el cwd de un notebook es su propia carpeta, no la raíz del repo.
X00T = stage00t_config_from_run(RUN_ID, project_root=ROOT)   # run + defaults de la etapa
RADIUS_PX      = float(X00T['a3_radius_px'])
THRESHOLD_PCT  = float(X00T['a3_threshold_pct'])
NEEDS_RED_CONT = bool(X00T['a3_science_needs_red_continuum'])
BANDS_A        = {k: tuple(v) for k, v in X00T['a3_bands_A'].items()}
PROTECTED_A    = [tuple(w) for w in X00T['a3_protected_windows_A']]
SIDE_WIDTH_A   = float(X00T['a3_side_width_A'])
GAP_A          = float(X00T['a3_gap_A'])

# La banda que A3 NO mide y que resulta ser el discriminante de §7: O₂ A es la
# más profunda del rango de MUSE y no está en `TELLURIC_BANDS` de la etapa.
O2_A_BAND = (7590.0, 7700.0)
BANDAS_MAS = dict(BANDS_A); BANDAS_MAS['O2_A'] = O2_A_BAND

# ---- a partir de aquí, cambia lo que quieras probar ----

_del_run = [k for k in X00T if k.startswith('a3_') and k in CFG]
for _k, _v in {'radio apertura (px)': RADIUS_PX, 'umbral (%)': THRESHOLD_PCT,
               'continuo rojo necesario': NEEDS_RED_CONT,
               'bandas': list(BANDS_A), 'protegidas': PROTECTED_A,
               'continuo lateral/hueco (Å)': (SIDE_WIDTH_A, GAP_A)}.items():
    print(f'  {_k:26s} {_v}')
print('\n  declaradas en el config del run:', _del_run or 'ninguna — todas son default de la etapa')


## 2 · Entradas — las tres reducciones y sus tres QC

El mismo objeto se ha reducido tres veces, y **cada reducción tiene su propio QC de A3 con su propio veredicto**. Esa es la primera mitad de la respuesta: la pregunta «¿cuánto vale la banda?» no tiene un número, tiene tres.

Los QC hermanos se **descubren**, no se nombran: se buscan los `stage00t*.json` de todos los runs del mismo objeto y se indexan por el cubo que cada uno declara. Pedirlos por nombre a `nb.load_qc` sería peor que inútil — si el fichero no existe en el run activo, la resolución cae al alias del registro y devuelve **el QC de otra reducción bajo el nombre del que pediste**, en silencio.

La celda dice también qué falta en disco: el cubo de entrada de la primera reducción **está borrado** (§10 lo reconstruye) y el «después» de la segunda vive **fuera** de su run.


In [ ]:
def qcs_de_a3():
    """Los QC de A3 de todos los runs de ESTE objeto, por su propio cubo."""
    out = {}
    for p in sorted((ROOT / 'runs').glob('*/stages/stage00t*.json')):
        run = p.parent.parent.name
        try:
            mismo = nb.run_target(run) == nb.run_target(RUN_ID)
        except Exception:
            mismo = False
        if not mismo:
            continue
        out[f'{run}/{p.name}'] = json.loads(p.read_text(encoding='utf-8'))
    return out

QC_A3 = qcs_de_a3()
# El de la cadena activa sí sale por la vía normal: así `chain.stage_runs['A3']`
# sigue mandando si algún día este objeto separa A3 en otro run.
QC_CAN = nb.load_qc_optional('stages/stage00t_qc.json', RUN_ID)

REDUCCIONES = []
for etiqueta, qc in QC_A3.items():
    dec = qc.get('decision', {}); ent = qc.get('input', {}); pro = qc.get('products', {})
    REDUCCIONES.append(dict(
        etiqueta=etiqueta,
        canonica=(QC_CAN is not None and qc.get('input', {}).get('sha256') == QC_CAN.get('input', {}).get('sha256')
                  and dec.get('verdict') == QC_CAN.get('decision', {}).get('verdict')),
        qc=qc,
        pre=resolver(ent.get('cube')),
        post=resolver(pro.get('cube_telcorr')),
        curva=resolver(pro.get('transmission')),
        yx=tuple(ent['primary_yx']) if ent.get('primary_yx') else None,
        radio=float(ent['aperture_radius_px']) if ent.get('aperture_radius_px') else None,
        aplicado=bool(dec.get('telluric_applied')),
        veredicto=dec.get('verdict'),
        profundidades=dec.get('depth_pct_by_band', {}),
    ))
REDUCCIONES.sort(key=lambda r: (r['canonica'], r['etiqueta']))

def _existe(p):
    return 'sí' if (p is not None and p.exists()) else ('BORRADO/ausente' if p else '—')

for r in REDUCCIONES:
    print(('· CANÓNICA  ' if r['canonica'] else '· histórica ') + r['etiqueta'])
    print(f"    veredicto {r['veredicto']}  aplicado={r['aplicado']}  "
          f"apertura={r['yx']} r={r['radio']}")
    print('    profundidades declaradas:', {k: round(v, 4) for k, v in r['profundidades'].items()})
    print(f"    cubo PRE   {_existe(r['pre']):16s} {r['pre']}")
    print(f"    cubo POST  {_existe(r['post']):16s} {r['post']}")
    print(f"    curva      {_existe(r['curva']):16s} {r['curva']}")

if QC_CAN is not None and not any(r['canonica'] for r in REDUCCIONES):
    print('\nAVISO: el QC de la cadena activa no se ha emparejado con ningún hermano.')
if not REDUCCIONES:
    print('A3 no ha corrido para este objeto: las secciones siguientes lo dirán y no inventarán nada.')


## 3 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal** del fuente, para que puedas editarla. Todo lo que viene después usa estos nombres locales, así que un cambio aquí se propaga al resultado — y la comparación del final lo cuantifica.

- `VerificationError` — de `musepipe/reduction/verify.py`
- `circular_aperture_mask` — de `musepipe/reduction/verify.py`
- `extract_aperture_spectrum` — de `musepipe/reduction/verify.py`
- `TelluricError` — de `musepipe/reduction/telluric.py`
- `TelluricDecision` — de `musepipe/reduction/telluric.py`
- `wavelength_axis_from_header` — de `musepipe/reduction/telluric.py`
- `window_mask` — de `musepipe/reduction/telluric.py`
- `protected_mask` — de `musepipe/reduction/telluric.py`
- `local_continuum_linear` — de `musepipe/reduction/telluric.py`
- `measure_telluric_depths` — de `musepipe/reduction/telluric.py`
- `decide_telluric` — de `musepipe/reduction/telluric.py`
- `enforce_protected_transmission` — de `musepipe/reduction/telluric.py`
- `validate_transmission_physical` — de `musepipe/reduction/telluric.py`
- `apply_transmission_to_arrays` — de `musepipe/reduction/telluric.py`
- `verify_outside_bands_unchanged` — de `musepipe/reduction/telluric.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from astropy.io import fits
from dataclasses import dataclass
from typing import Mapping
from typing import Sequence
import numpy as np
# Las dos excepciones viajan porque las levantan las funciones copiadas:
# un `raise` a una clase ausente reventaría el notebook al primer hueco.

HALPHA_PROTECTED = (6540.0, 6590.0)
NALGS_PROTECTED = (5780.0, 6050.0)
PROTECTED_WINDOWS = (HALPHA_PROTECTED, NALGS_PROTECTED)
TELLURIC_BANDS = {
    "O2_B": (6864.0, 6960.0),
    "H2O_7200": (7160.0, 7340.0),
    "H2O_8200": (8130.0, 8350.0),
}
DEFAULT_FIT_REGIONS = tuple(TELLURIC_BANDS.values())


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


class TelluricError(RuntimeError):
    """Raised when A3 must stop at a gate or checkpoint."""


@dataclass(frozen=True)
class TelluricDecision:
    depth_pct_by_band: dict[str, float]
    telluric_applied: bool
    science_needs_red_continuum: bool
    decision: str
    checkpoint_required: bool


def wavelength_axis_from_header(header: fits.Header, n_wave: int) -> np.ndarray:
    if all(key in header for key in ("CRVAL3", "CDELT3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CDELT3"])
    if all(key in header for key in ("CRVAL3", "CD3_3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CD3_3"])
    raise TelluricError("Could not recover wavelength axis from DATA header.")


def window_mask(wave: Sequence[float], window: tuple[float, float]) -> np.ndarray:
    wave_arr = np.asarray(wave, dtype=np.float64)
    lo, hi = window
    return (wave_arr >= lo) & (wave_arr <= hi) & np.isfinite(wave_arr)


def protected_mask(wave: Sequence[float], windows: Sequence[tuple[float, float]] = PROTECTED_WINDOWS) -> np.ndarray:
    mask = np.zeros(np.asarray(wave).shape, dtype=bool)
    for window in windows:
        mask |= window_mask(wave, window)
    return mask


def local_continuum_linear(
    wave: Sequence[float],
    spectrum: Sequence[float],
    band: tuple[float, float],
    *,
    side_width_A: float = 40.0,
    gap_A: float = 10.0,
) -> np.ndarray:
    """Interpolate a local continuum across one telluric band."""

    wave_arr = np.asarray(wave, dtype=np.float64)
    spec = np.asarray(spectrum, dtype=np.float64)
    lo, hi = band
    left = (wave_arr >= lo - gap_A - side_width_A) & (wave_arr <= lo - gap_A)
    right = (wave_arr >= hi + gap_A) & (wave_arr <= hi + gap_A + side_width_A)
    left &= np.isfinite(spec)
    right &= np.isfinite(spec)
    if not left.any() or not right.any():
        finite = np.isfinite(spec)
        fallback = float(np.nanmedian(spec[finite])) if finite.any() else 1.0
        return np.full(wave_arr.shape, fallback, dtype=np.float64)
    x = np.array([np.nanmedian(wave_arr[left]), np.nanmedian(wave_arr[right])], dtype=np.float64)
    y = np.array([np.nanmedian(spec[left]), np.nanmedian(spec[right])], dtype=np.float64)
    if not np.all(np.isfinite(y)) or x[0] == x[1]:
        return np.full(wave_arr.shape, float(np.nanmedian(spec[np.isfinite(spec)])), dtype=np.float64)
    return np.interp(wave_arr, x, y)


def measure_telluric_depths(
    wave: Sequence[float],
    spectrum: Sequence[float],
    *,
    bands: Mapping[str, tuple[float, float]] = TELLURIC_BANDS,
) -> dict[str, float]:
    """Measure telluric depth as percent drop relative to local continuum."""

    wave_arr = np.asarray(wave, dtype=np.float64)
    spec = np.asarray(spectrum, dtype=np.float64)
    depths: dict[str, float] = {}
    for name, band in bands.items():
        mask = window_mask(wave_arr, band)
        if not mask.any():
            depths[name] = float("nan")
            continue
        continuum = local_continuum_linear(wave_arr, spec, band)
        valid = mask & np.isfinite(spec) & np.isfinite(continuum) & (continuum != 0)
        if not valid.any():
            depths[name] = float("nan")
            continue
        ratio = spec[valid] / continuum[valid]
        depth = 100.0 * (1.0 - float(np.nanmedian(ratio)))
        depths[name] = max(0.0, depth)
    return depths


def decide_telluric(
    depth_pct_by_band: Mapping[str, float],
    *,
    science_needs_red_continuum: bool,
    threshold_pct: float = 3.0,
) -> TelluricDecision:
    depths = {str(key): float(value) for key, value in depth_pct_by_band.items()}
    finite_depths = [value for value in depths.values() if np.isfinite(value)]
    max_depth = max(finite_depths) if finite_depths else float("nan")
    if not science_needs_red_continuum:
        decision = "not_needed_science"
        applied = False
        checkpoint = False
    elif np.isfinite(max_depth) and max_depth < threshold_pct:
        decision = "not_needed_shallow"
        applied = False
        checkpoint = False
    elif np.isfinite(max_depth):
        decision = "needed"
        applied = True
        checkpoint = True
    else:
        decision = "depth_unknown_checkpoint"
        applied = False
        checkpoint = True
    return TelluricDecision(
        depth_pct_by_band=depths,
        telluric_applied=applied,
        science_needs_red_continuum=bool(science_needs_red_continuum),
        decision=decision,
        checkpoint_required=checkpoint,
    )


def enforce_protected_transmission(
    wave: Sequence[float],
    transmission: Sequence[float],
    *,
    windows: Sequence[tuple[float, float]] = PROTECTED_WINDOWS,
) -> np.ndarray:
    trans = np.asarray(transmission, dtype=np.float64).copy()
    if trans.ndim != 1:
        raise TelluricError("Transmission must be a 1D array.")
    mask = protected_mask(wave, windows)
    if mask.shape != trans.shape:
        raise TelluricError("Transmission and wavelength arrays must have the same shape.")
    trans[mask] = 1.0
    return trans


def validate_transmission_physical(transmission: Sequence[float]) -> bool:
    trans = np.asarray(transmission, dtype=np.float64)
    return bool(np.all(np.isfinite(trans)) and np.nanmin(trans) > 0.0 and np.nanmax(trans) <= 1.0)


def apply_transmission_to_arrays(
    data: np.ndarray,
    stat: np.ndarray,
    wave: Sequence[float],
    transmission: Sequence[float],
    *,
    min_transmission: float = 0.05,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Apply DATA/T and STAT/T^2, forcing protected windows to T=1."""

    cube = np.asarray(data, dtype=np.float64)
    variance = np.asarray(stat, dtype=np.float64)
    if cube.shape != variance.shape:
        raise TelluricError(f"DATA shape {cube.shape} != STAT shape {variance.shape}.")
    trans = enforce_protected_transmission(wave, transmission)
    if trans.shape[0] != cube.shape[0]:
        raise TelluricError("Transmission length does not match cube wavelength axis.")
    if np.nanmin(trans) < min_transmission:
        raise TelluricError(f"Transmission below safety floor {min_transmission}.")
    scale = trans[:, None, None]
    return cube / scale, variance / (scale**2), trans


def verify_outside_bands_unchanged(
    pre_spec: Sequence[float],
    post_spec: Sequence[float],
    wave: Sequence[float],
    *,
    max_change_pct: float = 0.2,
    corrected_bands: Sequence[tuple[float, float]] = DEFAULT_FIT_REGIONS,
) -> bool:
    pre = np.asarray(pre_spec, dtype=np.float64)
    post = np.asarray(post_spec, dtype=np.float64)
    wave_arr = np.asarray(wave, dtype=np.float64)
    mask = np.isfinite(pre) & np.isfinite(post) & (pre != 0)
    for band in corrected_bands:
        mask &= ~window_mask(wave_arr, band)
    mask &= ~protected_mask(wave_arr)
    if not mask.any():
        raise TelluricError("No outside-band channels available for verification.")
    change_pct = 100.0 * np.nanmedian(np.abs(post[mask] / pre[mask] - 1.0))
    return bool(change_pct <= max_change_pct)


## 4 · Chequeo de deriva

Compara el fuente copiado arriba con el que **hoy** tiene `musepipe`. Si alguien cambió la cadena, esta celda lo dice nombrando la función: es lo que evita que este notebook siga dando resultados «de la cadena» cuando ya no lo son.


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50",
    "musepipe/reduction/telluric.py:TelluricError": "5347f3394587",
    "musepipe/reduction/telluric.py:TelluricDecision": "854cf1dd3710",
    "musepipe/reduction/telluric.py:wavelength_axis_from_header": "f463214b3a31",
    "musepipe/reduction/telluric.py:window_mask": "f5bf804839d4",
    "musepipe/reduction/telluric.py:protected_mask": "3bd25d021885",
    "musepipe/reduction/telluric.py:local_continuum_linear": "342caee26693",
    "musepipe/reduction/telluric.py:measure_telluric_depths": "7ca4194d1d59",
    "musepipe/reduction/telluric.py:decide_telluric": "bd9e31d10459",
    "musepipe/reduction/telluric.py:enforce_protected_transmission": "f904d9698e66",
    "musepipe/reduction/telluric.py:validate_transmission_physical": "c1e41df83a9e",
    "musepipe/reduction/telluric.py:apply_transmission_to_arrays": "44fedc1b757c",
    "musepipe/reduction/telluric.py:verify_outside_bands_unchanged": "a84b16e93d26"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name),
                    None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} A3')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · El espectro ANTES y DESPUÉS de corregir

La figura que dice si la corrección se hizo bien. Para cada reducción que **sí** aplicó corrección se dibuja el espectro de la primaria antes y después, en el mismo eje: arriba el rango completo con las bandas sombreadas, abajo un panel por banda con el **continuo local que la etapa ajusta** (dos medianas laterales de `SIDE_WIDTH_A` Å separadas por un hueco de `GAP_A` Å, interpoladas) y la profundidad medida anotada.

**Cómo se lee:** la corrección está bien cuando el fondo de la banda **sube hasta la línea de continuo** y **nada fuera de las bandas se mueve**. Lo segundo no se deja al ojo: lo mide `verify_outside_bands_unchanged`, la V2 de la propia etapa, que exige ≤ 0.2 % de cambio fuera de las regiones corregidas.

En la reducción cuyo cubo de entrada está borrado el «antes» se **reconstruye** como `después × T` con la curva que la etapa guardó. No es una aproximación: `apply_transmission_to_cube_file` escribe la transmisión ya *forzada* (con las ventanas protegidas a 1), que es exactamente por la que dividió.

Se dibujan también las bandas que A3 **no** mide, del catálogo de `musepipe.telluric_lines` — entre ellas **O₂ A**, la más profunda del rango de MUSE. Que quede fuera de la decisión es el hilo del que tira §7.


In [ ]:
def espectro(cube_path, yx, radius, half=30):
    """(wave, spec) de una apertura, leyendo SOLO una ventana espacial.

    Los cubos pesan 1.4-3.5 GB y aquí solo hacen falta ~60x60 px alrededor de
    la primaria. `_read_primary_spectrum` de la etapa se trae el cubo entero;
    esto da el mismo espectro en ~2 s y sin llenar la memoria.
    """
    cube_path = Path(cube_path)
    with fits.open(cube_path, memmap=True) as h:
        hdu = h['DATA'] if 'DATA' in h else h[1 if h[0].data is None else 0]
        ny, nx = hdu.shape[1:]
        cy, cx = float(yx[0]), float(yx[1])
        y0 = max(0, int(np.floor(cy)) - half); y1 = min(ny, int(np.ceil(cy)) + half + 1)
        x0 = max(0, int(np.floor(cx)) - half); x1 = min(nx, int(np.ceil(cx)) + half + 1)
        win = np.asarray(hdu.data[:, y0:y1, x0:x1], dtype=np.float64)
        wave = wavelength_axis_from_header(hdu.header, hdu.shape[0])
        if not np.all(np.isfinite(wave)) or wave[0] == 0:
            wave = wavelength_axis_from_header(h[0].header, hdu.shape[0])
    # La apertura se recentra a coordenadas de la ventana.
    return wave, extract_aperture_spectrum(win, (cy - y0, cx - x0), radius)


def profundidad_cruda(wave, spec, banda):
    """`measure_telluric_depths` SIN el `max(0.0, depth)` — para ver el signo.

    Los ceros exactos que publica el QC pueden ser negativos recortados, y un
    residuo que cambia de signo no es una banda sin corregir: es una corregida.
    """
    m = window_mask(wave, banda)
    cont = local_continuum_linear(wave, spec, banda,
                                  side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
    v = m & np.isfinite(spec) & np.isfinite(cont) & (cont != 0)
    if not v.any():
        return float('nan')
    return 100.0 * (1.0 - float(np.nanmedian(spec[v] / cont[v])))


def transmision_en(curva, wave):
    with fits.open(Path(curva)) as h:
        tw = np.asarray(h[1].data['wave_A'], dtype=np.float64)
        tt = np.asarray(h[1].data['transmission'], dtype=np.float64)
    return np.interp(wave, tw, tt)


def par_antes_despues(r):
    """(wave, antes, despues, procedencia) de una reducción que corrigió."""
    if r['yx'] is None or r['radio'] is None:
        return None
    if r['pre'] is not None and r['pre'].exists() and r['post'] is not None and r['post'].exists():
        w, a = espectro(r['pre'], r['yx'], r['radio'])
        _, d = espectro(r['post'], r['yx'], r['radio'])
        return w, a, d, 'los dos cubos, en disco'
    if (r['post'] is not None and r['post'].exists()
            and r['curva'] is not None and r['curva'].exists()):
        w, d = espectro(r['post'], r['yx'], r['radio'])
        return w, d * transmision_en(r['curva'], w), d, 'ANTES reconstruido como después × T'
    return None


from musepipe.telluric_lines import TELLURIC_BANDS as CATALOGO, SEVERITY_ALPHA
COLOR_ESPECIE = {'O2': 'tab:orange', 'H2O': 'tab:cyan'}


def figura_antes_despues(w, antes, despues, titulo, procedencia):
    zooms = list(BANDAS_MAS.items())
    fig = plt.figure(figsize=(12.5, 7.2))
    gs = fig.add_gridspec(2, len(zooms), height_ratios=[1.7, 1.4], hspace=0.42, wspace=0.28)
    ax0 = fig.add_subplot(gs[0, :])
    for b in CATALOGO:
        ax0.axvspan(b['lo_A'], b['hi_A'], color=COLOR_ESPECIE.get(b['species'], 'grey'),
                    alpha=SEVERITY_ALPHA.get(b['severity'], 0.1), zorder=0)
        ax0.annotate(b['name'], ((b['lo_A'] + b['hi_A']) / 2, 0.985),
                     xycoords=('data', 'axes fraction'), ha='center', va='top',
                     fontsize=6, color='0.35')
    for lo, hi in PROTECTED_A:
        ax0.axvspan(lo, hi, color='0.5', alpha=0.16, zorder=0)
    ax0.plot(w, antes, lw=0.6, color='tab:red', label='antes (sin corregir)')
    ax0.plot(w, despues, lw=0.6, color='tab:blue', label='después (corregido)')
    ax0.set_xlim(w[0], w[-1])
    _fin = np.isfinite(antes)
    if _fin.any():
        ax0.set_ylim(np.nanpercentile(antes[_fin], 0.5), np.nanpercentile(antes[_fin], 99.8))
    ax0.set_xlabel('λ [Å]'); ax0.set_ylabel('flujo en la apertura')
    ax0.legend(loc='upper left', fontsize=8, framealpha=0.9)
    ax0.set_title(f'{titulo}\n({procedencia}; gris = ventanas protegidas)', fontsize=9)
    for j, (nombre, banda) in enumerate(zooms):
        ax = fig.add_subplot(gs[1, j])
        lo, hi = banda
        sel = (w >= lo - 2.2 * SIDE_WIDTH_A) & (w <= hi + 2.2 * SIDE_WIDTH_A)
        ax.axvspan(lo, hi, color='tab:orange', alpha=0.15, zorder=0)
        for datos, color, etq in ((antes, 'tab:red', 'antes'), (despues, 'tab:blue', 'después')):
            ax.plot(w[sel], datos[sel], lw=0.7, color=color, label=etq)
            cont = local_continuum_linear(w, datos, banda,
                                          side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
            ax.plot(w[sel], cont[sel], lw=0.9, ls='--', color=color, alpha=0.65)
        d_a = profundidad_cruda(w, antes, banda); d_d = profundidad_cruda(w, despues, banda)
        ax.set_title(f'{nombre}\nantes {d_a:.2f} % · después {d_d:.2f} %', fontsize=8)
        ax.set_xlim(w[sel][0], w[sel][-1]); ax.tick_params(labelsize=7)
        ax.set_xlabel('λ [Å]', fontsize=7)
        if j == 0:
            ax.legend(fontsize=6.5, loc='lower left')
    # Sin `tight_layout`: el gridspec ya fija los huecos, y mezclarlos avisa.
    plt.show()


PARES = {}
for r in REDUCCIONES:
    if not r['aplicado']:
        continue
    par = par_antes_despues(r)
    if par is None:
        print('sin par antes/después para', r['etiqueta'],
              '— falta el cubo o la curva que declara su QC')
        continue
    w, a, d, proc = par
    PARES[r['etiqueta']] = (w, a, d)
    v2 = verify_outside_bands_unchanged(a, d, w)
    print(f"{r['etiqueta']}: {proc}")
    print(f"   V2 (fuera de banda sin tocar, ≤0.2 %): {'PASA' if bool(v2) else 'FALLA'}")
    figura_antes_despues(w, a, d, f"{TARGET} · {r['etiqueta']}", proc)
if not PARES:
    print('ninguna reducción de este objeto aplicó corrección: no hay antes/después que enseñar')


## 6 · El cubo canónico, y la apertura que su QC no declara

El QC multi-noche **no registra ni `primary_yx` ni `aperture_radius_px`** — el esquema nuevo los perdió. Sin ellos su 0.586 % no se puede reproducir sin adivinar, así que en vez de esconder el hueco se barre: posiciones candidatas (el centro del recorte de B1 redondeado, el centro sin redondear, el pico de luz blanca) × radios, y se marca la celda que da el número del QC.

Ojo con el marco: la primaria de B3 está en coordenadas del **cubo recortado** (170 px), y el cubo canónico es el de 200 px. El desfase lo declara `stage01_qc.json:crop_bounds_per_cube` y confundirlos mueve la apertura 15 px.

Debajo, las profundidades **sin recortar a cero**. Los `0.0` que publica el QC no son bandas planas: son valores negativos pasados por `max(0.0, depth)`. Un residuo que se va por debajo del continuo tanto como por encima es la firma de una banda **ya corregida**, no de una que nadie tocó.


In [ ]:
CAN = next((r for r in REDUCCIONES if r['canonica']), None)
if CAN is None:
    print('no hay QC de A3 para la cadena activa: nada que reproducir aquí')
else:
    qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
    centro = [float(v) for v in qc01['crop']['center_yx']]
    bounds = (qc01.get('crop_bounds_per_cube') or [{}])[0]
    off = (float(bounds.get('y1', 0)), float(bounds.get('x1', 0)))
    print(f'centro del recorte (marco del cubo entero): {centro}  ·  desfase B1: {off}')

    with fits.open(CAN['pre'], memmap=True) as h:
        _hdu = h['DATA'] if 'DATA' in h else h[1]
        _cy, _cx = int(round(centro[0])), int(round(centro[1]))
        _w = np.asarray(_hdu.data[::37, _cy - 30:_cy + 31, _cx - 30:_cx + 31], dtype=np.float64)
    _med = np.nanmedian(_w, axis=0)
    _pk = np.unravel_index(np.nanargmax(_med), _med.shape)
    pico = (_cy - 30 + int(_pk[0]), _cx - 30 + int(_pk[1]))

    CAN_YX = (round(centro[0]), round(centro[1]))
    objetivo = CAN['profundidades'].get('O2_B')
    print(f'\nbarrido de O₂ B (%). Objetivo del QC: {objetivo!r}\n')
    radios = [4.0, 6.0, 8.0, 10.0, 12.0]
    print('  posición'.ljust(34) + ''.join(f'r={r:<9.0f}' for r in radios))
    for pos, etq in ((CAN_YX, 'round(centro)'), (tuple(centro), 'centro'), (pico, 'pico luz blanca')):
        fila = ''
        for r_ in radios:
            _, sp = espectro(CAN['pre'], pos, r_)
            v = measure_telluric_depths(_, sp, bands=BANDS_A)['O2_B']
            marca = ' <=' if (objetivo is not None and np.isclose(v, objetivo, rtol=1e-9)) else '   '
            fila += f'{v:8.4f}{marca}'
        print(f'  {etq:16s} {str(tuple(round(float(c), 2) for c in pos)):15s}' + fila)

    WAVE_CAN, SPEC_CAN = espectro(CAN['pre'], CAN_YX, RADIUS_PX)
    print(f'\napertura recuperada: yx={CAN_YX} r={RADIUS_PX}  '
          f'(el QC no declara ninguna de las dos)')
    print('profundidades SIN recortar a cero:')
    for nombre, banda in BANDAS_MAS.items():
        cruda = profundidad_cruda(WAVE_CAN, SPEC_CAN, banda)
        publicada = CAN['profundidades'].get(nombre)
        nota = '  <- negativa, el QC publica 0.0' if cruda < 0 else ''
        extra = '' if publicada is not None else '  (no la mide la etapa)'
        print(f'  {nombre:9s} {cruda:8.3f} %   publicada: {publicada}{extra}{nota}')


### La misma banda en las tres reducciones, normalizada a su continuo

Los tres cubos tienen escalas de flujo distintas, así que para verlos juntos se divide cada espectro **por su propio continuo local** — el mismo que ajusta la etapa. En ese eje, `1.0` es «no hay banda» y el fondo del hueco es la transmisión.

Es la lectura directa de si la reducción quedó bien: **si el canónico se parece al «después» de la reducción antigua y no a su «antes», la corrección está hecha** — y entonces su 0.59 % es un residuo, no una banda intacta.


In [ ]:
if CAN is None or not PARES:
    print('hacen falta el cubo canónico y al menos una reducción con antes/después')
else:
    zooms = list(BANDAS_MAS.items())
    fig, axes = plt.subplots(1, len(zooms), figsize=(13.5, 3.4), squeeze=False)
    etq_ref, (w_ref, a_ref, d_ref) = next(iter(PARES.items()))
    for ax, (nombre, banda) in zip(axes[0], zooms):
        lo, hi = banda
        ax.axvspan(lo, hi, color='tab:orange', alpha=0.15, zorder=0)
        ax.axhline(1.0, color='0.55', lw=0.8, ls=':', zorder=1)
        for w_, sp, color, lab in ((w_ref, a_ref, 'tab:red', 'antigua · antes'),
                                   (w_ref, d_ref, 'tab:blue', 'antigua · después'),
                                   (WAVE_CAN, SPEC_CAN, 'k', 'canónica (multi-noche)')):
            cont = local_continuum_linear(w_, sp, banda,
                                          side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
            sel = (w_ >= lo - 1.2 * SIDE_WIDTH_A) & (w_ <= hi + 1.2 * SIDE_WIDTH_A)
            with np.errstate(invalid='ignore', divide='ignore'):
                razon = np.where(cont != 0, sp / cont, np.nan)
            ax.plot(w_[sel], razon[sel], lw=0.7, color=color, label=lab)
        ax.set_title(nombre, fontsize=9); ax.tick_params(labelsize=7)
        ax.set_xlabel('λ [Å]', fontsize=7); ax.set_ylim(0.45, 1.35)
    axes[0][0].set_ylabel('flujo / continuo local', fontsize=8)
    axes[0][0].legend(fontsize=6.5, loc='lower left')
    fig.suptitle(f'{TARGET} · la misma banda en las tres reducciones '
                 f'(histórica: {etq_ref})', fontsize=9)
    fig.tight_layout(); plt.show()


## 7 · Hipótesis 1 — el DRS ya las quitó (la que responde)

Si el cubo canónico ya viene con la corrección telúrica aplicada **por exposición**, su 0.59 % no es la profundidad de la banda: es lo que **queda** después de corregir. Y eso se comprueba sin abrir un cubo, mirando qué se le dio a `muse_scipost`.

El SOF es la lista de entradas de cada receta. `STD_TELLURIC` es la tabla de absorción telúrica medida sobre la estrella estándar de esa noche: si está, el DRS divide por ella; si no está, no. La cadena multi-noche la **exige** — `musepipe/reduction/perexp_plan.py` la añade al plan y levanta `PerExposurePlanError` si las cuentas no salen 1/1 por exposición.

El número que decide es **O₂ A**, la banda que `decide_telluric` no mira. Es la más profunda del rango: si las bandas siguieran ahí, ahí se vería.


In [ ]:
def censo_sof(carpeta, patron='muse_scipost*.sof'):
    """Cuántos SOF hay y cuántas veces aparece cada etiqueta de calibración."""
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        return None
    n, tags = 0, {'STD_RESPONSE': 0, 'STD_TELLURIC': 0}
    for sof in sorted(carpeta.glob(patron)):
        n += 1
        etiquetas = [ln.split()[-1] for ln in sof.read_text(encoding='utf-8').splitlines() if ln.strip()]
        for t in tags:
            tags[t] += etiquetas.count(t)
    return dict(n_sof=n, **tags)


print('=== qué se le dio a muse_scipost en cada reducción ===')
for r in REDUCCIONES:
    if r['pre'] is None:
        continue
    # El workdir de la reducción se deduce del cubo que el QC declara:
    #   .../<workdir>/final/DATACUBE_FINAL.fits   (cadena multi-noche)
    #   .../raw_reduction/products/<receta>/DATACUBE_FINAL.fits  (antiguas)
    candidatos = [r['pre'].parent.parent / 'sof',
                  r['pre'].parent.parent.parent / 'sof',
                  r['pre'].parent.parent / 'cubes' / 'sof']
    for c in candidatos:
        censo = censo_sof(c)
        if censo and censo['n_sof']:
            marca = 'CORRIGE tellúrico' if censo['STD_TELLURIC'] else 'NO corrige telúrico'
            print(f"  {r['etiqueta']:44s} {censo['n_sof']:3d} SOF · "
                  f"STD_RESPONSE {censo['STD_RESPONSE']:3d} · STD_TELLURIC {censo['STD_TELLURIC']:3d}  -> {marca}")
            print(f'      {c}')
            break
    else:
        print(f"  {r['etiqueta']:44s} sin SOF en disco (workdir borrado)")

print('\n=== O₂ A (7590-7700 Å), la banda que A3 NO mide ===')
for etiqueta, (w, a, d) in PARES.items():
    print(f'  {etiqueta:44s} antes {profundidad_cruda(w, a, O2_A_BAND):7.3f} %'
          f'   después {profundidad_cruda(w, d, O2_A_BAND):7.3f} %')
if CAN is not None:
    print(f"  {CAN['etiqueta']:44s} {profundidad_cruda(WAVE_CAN, SPEC_CAN, O2_A_BAND):7.3f} %"
          '   (la cadena canónica no aplicó nada)')


## 8 · Hipótesis 2 — combinar 29 exposiciones a masas de aire distintas

La absorción telúrica crece con la masa de aire: por Beer–Lambert, una banda con transmisión `T₀` a `X₀` tiene `T = T₀^(X/X₀)` a otra `X`. Promediar exposiciones tomadas a masas de aire distintas mezcla profundidades distintas, y podría difuminar la banda.

Se mide, no se supone: las masas de aire salen de las **cabeceras de los originales** (`ESO TEL AIRM START/END`), a través del índice de exposiciones del workdir. Son lecturas de cabecera, no de datos: cuestan décimas de segundo.

La cota se calcula con la banda más profunda medida, llevada exposición a exposición y promediada con el peso real de la combinación (`EXPTIME`). Si el factor que sale es mucho menor que el observado, esta hipótesis no explica nada — y conviene mirar su **signo**.


In [ ]:
def masas_de_aire(workdir):
    """(exposición, noche, X, EXPTIME) leyendo SOLO cabeceras de los originales."""
    idx = Path(workdir) / 'inputs' / 'exposures.json'
    if not idx.exists():
        return []
    data = json.loads(idx.read_text(encoding='utf-8'))
    exps = data['exposures'] if isinstance(data, dict) else data
    filas = []
    for e in exps:
        crudo = (e.get('metadata') or {}).get('raw_object')
        if not crudo or not Path(crudo).exists():
            continue
        h = fits.getheader(crudo, 0)
        x0 = float(h.get('ESO TEL AIRM START', np.nan))
        x1 = float(h.get('ESO TEL AIRM END', np.nan))
        filas.append(dict(exp=e.get('exposure_id', Path(crudo).stem),
                          noche=str(h.get('DATE-OBS', ''))[:10],
                          x=0.5 * (x0 + x1), x0=x0, x1=x1,
                          t=float(h.get('EXPTIME', np.nan))))
    return filas


FILAS = masas_de_aire(CAN['pre'].parent.parent) if CAN is not None else []
if not FILAS:
    print('sin índice de exposiciones en disco: no se puede medir la masa de aire')
else:
    X = np.array([f['x'] for f in FILAS]); T_EXP = np.array([f['t'] for f in FILAS])
    print(f'{len(FILAS)} exposiciones · X de {X.min():.3f} a {X.max():.3f} · '
          f'media {X.mean():.3f} · ponderada por EXPTIME {np.average(X, weights=T_EXP):.3f}')
    for n in sorted({f['noche'] for f in FILAS}):
        sub = [f for f in FILAS if f['noche'] == n]
        print(f"  {n}: {len(sub):2d} exp · EXPTIME {sorted({int(s['t']) for s in sub})} · "
              f"X {min(s['x0'] for s in sub):.3f} -> {max(s['x1'] for s in sub):.3f}")

    fig, ax = plt.subplots(figsize=(9, 3.4))
    for n in sorted({f['noche'] for f in FILAS}):
        idx = [i for i, f in enumerate(FILAS) if f['noche'] == n]
        ax.scatter(idx, [FILAS[i]['x'] for i in idx],
                   s=[max(6.0, FILAS[i]['t'] / 8.0) for i in idx], label=n)
    ax.set_xlabel('exposición (orden del plan de combinación)')
    ax.set_ylabel('masa de aire X')
    ax.set_title('masa de aire por exposición (tamaño = EXPTIME)', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.25)
    fig.tight_layout(); plt.show()

    # Cota de Beer-Lambert: la banda más profunda medida, llevada a cada X.
    prof = [(e, profundidad_cruda(w, a, O2_A_BAND) / 100.0) for e, (w, a, _d) in PARES.items()]
    prof = [(e, p) for e, p in prof if np.isfinite(p) and p > 0]
    if not prof:
        print('sin banda de referencia medida: no se puede acotar')
    else:
        etq_ref, d0 = max(prof, key=lambda kv: kv[1])
        ref = next(r for r in REDUCCIONES if r['etiqueta'] == etq_ref)
        x0 = float((ref['qc'].get('fit') or {}).get('airmass_sci') or np.average(X, weights=T_EXP))
        d_comb = 1.0 - np.average((1.0 - d0) ** (X / x0), weights=T_EXP)
        obs = None
        if CAN is not None:
            obs = profundidad_cruda(WAVE_CAN, SPEC_CAN, O2_A_BAND) / 100.0
        print(f'\nreferencia: O₂ A = {100*d0:.2f} % en {etq_ref} (X₀ = {x0:.3f})')
        print(f'combinada sobre las {len(FILAS)} exposiciones reales: {100*d_comb:.2f} %'
              f'  ->  factor {d_comb/d0:.3f}x')
        if obs is not None and obs > 0:
            print(f'factor observado en el cubo canónico: {obs/d0:.4f}x  (1/{d0/obs:.1f})')
            print('la masa de aire ' + ('NO explica la caída: va en el sentido contrario'
                                       if d_comb >= d0 else 'no basta para explicar la caída'))


## 9 · Hipótesis 3 — la mediana sobre toda la banda diluye el núcleo

`measure_telluric_depths` compara la **mediana** de la banda entera con su continuo. Una banda con un núcleo estrecho y profundo entre alas transparentes da una mediana pequeña: el estimador diluye. Es real y explica por qué A3 puede llamar «superficial» a una banda cuyo fondo baja mucho más — pero se cuantifica poniendo la mediana al lado del percentil 10 y del mínimo.


In [ ]:
def dispersion_en_banda(wave, spec, banda):
    m = window_mask(wave, banda)
    cont = local_continuum_linear(wave, spec, banda,
                                  side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
    v = m & np.isfinite(spec) & np.isfinite(cont) & (cont != 0)
    if not v.any():
        return None
    caida = 100.0 * (1.0 - spec[v] / cont[v])
    return dict(mediana=float(np.nanmedian(caida)), p90=float(np.nanpercentile(caida, 90)),
                maximo=float(np.nanmax(caida)), n=int(v.sum()))


if CAN is not None:
    print(f"{'banda':10s} {'mediana':>9s} {'p90':>9s} {'máximo':>9s}   (% de caída bajo el continuo)")
    print(f"-- cubo canónico ({CAN['etiqueta']})")
    for nombre, banda in BANDAS_MAS.items():
        d = dispersion_en_banda(WAVE_CAN, SPEC_CAN, banda)
        if d:
            print(f"{nombre:10s} {d['mediana']:9.3f} {d['p90']:9.3f} {d['maximo']:9.3f}")
for etiqueta, (w, a, _d) in PARES.items():
    print(f'-- antes de corregir ({etiqueta})')
    for nombre, banda in BANDAS_MAS.items():
        d = dispersion_en_banda(w, a, banda)
        if d:
            print(f"{nombre:10s} {d['mediana']:9.3f} {d['p90']:9.3f} {d['maximo']:9.3f}")
print('\nLa mediana es el estimador de la etapa; el máximo es lo que se ve en la figura de §5.')


## 10 · Comparación con la cadena

Tres reproducciones, una por reducción, cada una contra **su** QC:

1. **canónica** — `measure_telluric_depths` sobre el cubo multi-noche con las perillas resueltas y la apertura recuperada en §6, contra `depth_pct_by_band` a `rtol=1e-9`; y `decide_telluric` contra el veredicto;
2. **con los dos cubos en disco** — antes y después, contra las profundidades declaradas y contra la V1 del QC, a la precisión con que el QC las guardó;
3. **con el cubo de entrada borrado** — el «antes» reconstruido como `después × T`.

`IDÉNTICO` solo sale si pasan **todas** las comparaciones disponibles. Si falta un cubo se dice cuál y no se imprime: un hueco no es un acuerdo.


In [ ]:
def _tolerancia(valor):
    """El QC guarda algunos números redondeados: se compara a SU precisión."""
    txt = repr(float(valor))
    dec = len(txt.split('.')[1]) if '.' in txt else 0
    return 0.5 * 10.0 ** (-dec) if dec <= 6 else 0.0


def compara(nombre, mio, suyo, rtol=1e-9):
    if suyo is None:
        print(f'    {nombre:26s} el QC no lo declara'); return True
    tol = _tolerancia(suyo)
    ok = bool(np.isclose(float(mio), float(suyo), rtol=rtol, atol=tol))
    print(f'    {nombre:26s} {float(mio):12.6f}  QC {float(suyo):12.6f}'
          f"{'' if ok else '   <-- DIFIERE'}")
    return ok


ok = True; comparadas = 0
for r in REDUCCIONES:
    print(('· CANÓNICA  ' if r['canonica'] else '· histórica ') + r['etiqueta'])
    if r['canonica']:
        if CAN is None or 'WAVE_CAN' not in globals():
            print('    sin cubo: no se compara'); ok = False; continue
        mias = measure_telluric_depths(WAVE_CAN, SPEC_CAN, bands=BANDS_A)
        for banda, suyo in r['profundidades'].items():
            ok &= compara(banda, mias.get(banda, float('nan')), suyo); comparadas += 1
        dec = decide_telluric(mias, science_needs_red_continuum=NEEDS_RED_CONT,
                              threshold_pct=THRESHOLD_PCT)
        suyo = r['qc'].get('decision', {})
        for campo, mio, ref in (('veredicto', dec.decision, suyo.get('verdict')),
                                ('aplicado', dec.telluric_applied, suyo.get('telluric_applied')),
                                ('checkpoint', dec.checkpoint_required, suyo.get('checkpoint_required'))):
            igual = (ref is None) or (mio == ref)
            print(f"    {campo:26s} {str(mio):>12s}  QC {str(ref):>12s}"
                  f"{'' if igual else '   <-- DIFIERE'}")
            ok &= igual; comparadas += 1
        continue
    par = PARES.get(r['etiqueta'])
    if par is None:
        print('    sin par antes/después en disco: no se compara'); ok = False; continue
    w, antes, despues = par
    mias = measure_telluric_depths(w, antes, bands=BANDS_A)
    for banda, suyo in r['profundidades'].items():
        ok &= compara('antes · ' + banda, mias.get(banda, float('nan')), suyo); comparadas += 1
    v1 = (r['qc'].get('verification') or {}).get('v1_o2_depth_pre_post_pct')
    if isinstance(v1, (list, tuple)) and len(v1) == 2:
        post = measure_telluric_depths(w, despues, bands=BANDS_A)
        ok &= compara('después · O2_B (V1)', post.get('O2_B', float('nan')), v1[1]); comparadas += 1

print()
if ok and comparadas:
    print('IDÉNTICO: la copia reproduce la cadena.')
else:
    print('DIFIERE — si has tocado una perilla, es lo esperado; '
          'si no, revisa el chequeo de deriva y qué cubos faltan.')


## 11 · Conclusión

**El 0.59 % del cubo canónico no es la profundidad de la banda O₂ B: es lo que queda después de corregirla.** Las tres hipótesis, por lo que mide este notebook:

1. **El DRS ya las quitó — es la respuesta.** La cadena multi-noche le pasa `STD_TELLURIC` a `muse_scipost` en cada exposición (y el planificador lo exige, no es suerte); las reducciones antiguas no se lo pasaban. La prueba independiente es **O₂ A**, que A3 no mide y que cae de ~29 % a ~1 % entre una reducción y otra.
2. **La masa de aire contribuye, pero no explica.** Con las 29 exposiciones reales y su peso por `EXPTIME`, Beer–Lambert da un factor de orden 1 — y del **signo contrario**: combinar a masas de aire mayores dejaría la banda *algo más profunda*, no veinte veces menos.
3. **La mediana diluye, y por eso el veredicto es coherente.** El estimador de la etapa mide la mediana de la banda, no su núcleo; es lo que permite que una banda con caídas locales mayores dé un número por debajo del 3 %.

### Lo que la cadena puede afirmar ahora

Que la corrección telúrica del cubo canónico **está hecha, por el DRS, exposición a exposición y cada una a su propia masa de aire** — que es mejor que aplicar una curva única al cubo combinado, como hacían las reducciones antiguas. El `not_needed_shallow` de A3 es entonces la lectura correcta de un residuo, y no una banda sin tratar. El residuo, medido aquí, es de ~0.6 % en O₂ B y ~1.3 % en O₂ A.

### Dos defectos que este notebook destapa

- **El QC multi-noche no declara la apertura** (`primary_yx`, `aperture_radius_px`): sus números no son reproducibles a partir del QC solo. Aquí se recuperan por barrido, pero eso es arqueología, no procedencia.
- **A3 no mide O₂ A**, la banda más profunda del rango. Su decisión se toma sobre tres bandas que no incluyen la que más informa. No se cambia aquí — es una decisión científica de la etapa — pero queda medido lo que costaría verlo.
